# FACE schizophrenia cohort: a first tutorial

This notebook shows how to load the FACE SZ data with the `facedata` package and how to take a first look at them. It is written for people who have never used this cohort and who do not code much.

You will learn how to:

1. load the data and understand the six tables you get;
2. find a variable and read what it means;
3. look at rating scales, lab results, neuropsychology, treatments and history at the inclusion visit;
4. follow a few patients over time, and the whole cohort over time;
5. judge how much a value can be trusted;
6. avoid the traps of this cohort;
7. build your own table and save it;
8. **put the bipolar and the schizophrenia cohorts in one table.**

**How to use a notebook.** A notebook is made of cells. Click in a grey cell and press `Shift` + `Enter` to run it: the result appears under it. Run the cells from top to bottom, because each one uses what the cells above it made. You can change a cell and run it again as often as you like; nothing you do here changes the data file.

## Before you start: these are patient data

The results that appear under the cells are patient data, even if the patients have no name.

- **Before you share this notebook or send it to GitHub, clear the results:** menu *Edit*, then *Clear Outputs of All Cells*, then save.
- Never copy patient rows into an e-mail, a chat or a slide. Share counts, means and figures of groups.
- Save your own tables in the folder `data/exports/`. Everything in `data/` stays on this computer.

## 1. Load the cohort

The first cell brings in the tools. The second one reads the extraction: it takes about half a minute, so wait for the number in brackets on the left of the cell to appear.

If your extraction file has another name, change it in the first cell.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from facedata import sz

EXTRACTION = "data/sz/SZALL_20261113.tsv"          # the file you received
pd.set_option("display.max_columns", 30)

In [ ]:
result = sz.load(EXTRACTION)

`result` holds six tables. You will mostly use the first two.

| Table | One row per | What it holds |
|---|---|---|
| `result.data` | patient visit | everything measured at a visit: scales, lab results, tests, answers |
| `result.patients` | patient | what does not change: sex, diagnosis, family history |
| `result.treatments` | patient and medication | each medication with its start and end dates |
| `result.relatives` | patient and relative | the family history, relative by relative |
| `result.qc` | variable | the catalogue: every variable, its label and how well it is filled |
| `result.flags` | flagged cell | every value that was repaired, doubted or removed, with the reason |

Let us give the first two a short name and count.

In [ ]:
data = result.data
patients = result.patients

print(len(data), "rows of", data["usubjid"].nunique(), "patients")
print(data.shape[1], "columns in data and", patients.shape[1], "in patients")

## 2. What a visit looks like

Every row of `data` starts with the same columns:

- `cohort`: `sz` on every row. It matters when you put two cohorts together (section 15).
- `usubjid`: the patient's number. It links all the tables.
- `visit`: the name of the visit. `screening` comes before inclusion, `V0` is the inclusion visit, `V1_an` is one year later, `V2_ans` two years later, and so on. This cohort has no six-month visit.
- `visit_month`: the planned number of months since inclusion (0, 12, 24...). Easier to plot than the names.
- `visit_date`, `siteid` (the centre), `age` (at the visit).

The cell below shows the first rows, with two scores next to these columns.

In [ ]:
data[["cohort", "usubjid", "visit", "visit_month", "visit_date", "siteid", "age", "panss_total", "cgi_severity"]].head()

How many visits of each kind are there? `value_counts()` counts the rows for each value, and `.plot.bar()` draws the counts.

In [ ]:
data["visit"].value_counts().plot.bar(figsize=(10, 3), title="Number of visits of each kind")
plt.show()

Most analyses start from the inclusion visit. The next cell keeps only the rows where `visit` is `V0`, and counts the inclusions per year.

**The pattern to remember:** `table[table["column"] == value]` keeps the rows where the column has that value.

In [ ]:
inclusion = data[data["visit"] == "V0"]

inclusion["visit_date"].dt.year.value_counts().sort_index().plot.bar(figsize=(10, 3), title="Inclusion visits per year")
plt.show()

## 3. Who the patients are

Sex, diagnosis and age at inclusion are in `patients`, one row per patient.

In [ ]:
print(patients["sex"].value_counts())
print()
print(patients["diagnosis"].value_counts())

patients["age_at_inclusion"].plot.hist(bins=30, title="Age at inclusion (years)")
plt.show()

To use these columns together with a visit, join the two tables on the patient's number with `merge`. From now on `inclusion` also has the sex and the diagnosis.

In [ ]:
inclusion = inclusion.merge(patients[["usubjid", "sex", "diagnosis"]], on="usubjid")
inclusion.groupby("diagnosis")["age"].agg(["count", "mean"]).round(1)

## 4. Find a variable

There are almost two thousand variables. `result.qc` is their catalogue: the name to type, a label, the instrument it belongs to, the dictionary tab, and `n_valid`, the number of values that can be used.

The cell below searches the labels for a word. Change `"clozapine"` to anything: `"sleep"`, `"suicide"`, `"cholesterol"`...

In [ ]:
catalogue = result.qc[["variable", "label", "instrument", "tab", "type", "unit", "level", "n_valid"]]

catalogue[catalogue["label"].str.contains("clozapine", case=False)]

The column `level` tells you in which table the variable is: `visit` means `data`, `patient` means `patients`, `list` means `treatments` and `relatives` means `relatives`.

You can also list everything one instrument holds, or count the instruments of each tab.

In [ ]:
print(catalogue.groupby("tab")["instrument"].nunique())

catalogue[catalogue["instrument"] == "psp"]

**Read the notes of an instrument before you use it.** Each instrument has a short text that says what was asked, how the score is computed and what to be careful about. This is where the traps are written. For the PANSS, the notes say which factor scores were corrected, and why.

In [ ]:
from facedata.schema import load_schemas

notes = load_schemas(sz.SCHEMAS)
print(notes["panss"]["notes"])

## 5. Rating scales and questionnaires

Five scores at the inclusion visit: the PANSS total and its positive and negative scales (symptoms, rated by the clinician), the CGI severity (1 to 7) and the PSP (functioning, 0 to 100, higher is better).

`describe()` gives the number of values, the mean, the spread and the quartiles.

In [ ]:
scores = ["panss_total", "panss_positive", "panss_negative", "cgi_severity", "psp_total"]

inclusion[scores].describe().round(1)

A histogram shows how the values are spread. These scores are whole numbers, so a few bars look too low or too high only because of where the bars are cut: change `bins=30` to another number and run the cell again to see.

In [ ]:
inclusion[scores].hist(bins=30, figsize=(10, 6))
plt.show()

Do patients with more negative symptoms function less well? Each dot is one patient at inclusion.

In [ ]:
inclusion.plot.scatter(x="panss_negative", y="psp_total", alpha=0.2, title="Negative symptoms (PANSS) against functioning (PSP)")
plt.show()

inclusion[["panss_negative", "psp_total"]].corr().round(2)

Compare groups with `groupby`: the mean of each score for each diagnosis, then a box plot of the PANSS total.

In [ ]:
print(inclusion.groupby("diagnosis")[scores].mean().round(1))

inclusion.boxplot(column="panss_total", by="diagnosis")
plt.show()

**Two things to know about these scales.**

- The PANSS also comes as five-factor scores (`panss_wallwork_positive`, `panss_lancon_negative`...). Three of them were computed with a wrong item in the database; the columns you get here follow the publications.
- The depression scale of Calgary has no total: its item 8 is missing from the extraction. `calgary_sum8` is the sum of the eight items that exist, so the usual threshold of 6 does not apply to it.

The self-questionnaires work the same way: `birchwood_total` (insight), `sqol_index` (quality of life), `mars_total` (adherence), `ctq_total` (childhood trauma), `psqi_total` (sleep).

In [ ]:
inclusion[["calgary_sum8", "birchwood_total", "sqol_index", "mars_total", "ctq_total", "psqi_total"]].describe().round(1)

## 6. Lab results

Lab results arrive from many laboratories, in several units. The package brings each one to a single unit, shown in the catalogue. Vitamin D is the exception: it has no unit, and one centre seems to report on another scale.

In [ ]:
labs = ["glucose", "crp", "prolactin", "vitamin_d", "clozapine"]

catalogue[catalogue["variable"].isin(labs)]

In [ ]:
inclusion[labs].describe().round(2)

**Results below the detection limit.** A laboratory sometimes answers "less than 5" instead of a number. The value then holds the limit (5), and a second column ending in `_cens` says so: `-1` means "the true value is below", `0` means measured, `1` means "above".

In [ ]:
inclusion["crp_cens"].value_counts()

Two pictures: prolactin for men and women, and the clozapine level of the patients who were on clozapine on the day of the visit (`on_clozapine` is explained in section 8).

In [ ]:
figure, (left, right) = plt.subplots(1, 2, figsize=(11, 3.5))

inclusion.boxplot(column="prolactin", by="sex", ax=left)
left.set_title("Prolactin (ng/mL)")
inclusion[inclusion["on_clozapine"] == 1]["clozapine"].plot.hist(bins=30, ax=right, title="Clozapine level of patients on clozapine")
figure.suptitle("")
plt.show()

Body mass index, cut into the usual classes with `pd.cut`.

In [ ]:
bmi_class = pd.cut(inclusion["bmi"], [0, 18.5, 25, 30, 100], labels=["under 18.5", "18.5 to 25", "25 to 30", "30 and more"])

bmi_class.value_counts(sort=False).plot.bar(title="Body mass index at inclusion")
plt.show()

## 7. Neuropsychology

The whole battery is given at inclusion, at three years and at five years. At one year only a few tests are done, and almost none at two and four years.

Many results come as a **z-score**: 0 is the average of people of the same age in the general population, and here higher is always better. A z-score of -1 is one standard deviation below that average.

In [ ]:
print(data[data["tmt_a_time"].notna()]["visit"].value_counts())
print()
print(data[data["cvlt_total_1_5"].notna()]["visit"].value_counts())

tests = ["tmt_a_time_z", "tmt_b_time_z", "wais4_ds_z", "wais4_mat_z", "wais4_simil_z", "cvlt_total_1_5_z"]

inclusion[tests].plot.box(figsize=(11, 4), title="Z-scores at inclusion (0 is the population average)")
plt.axhline(0, color="grey")
plt.show()

**Two editions of the same test are never mixed.** The centres moved from WAIS-III to WAIS-IV in 2015. The two editions do not count the same way, so each has its own columns (`wais3_...`, `wais4_...`), and `wais_version` says which one was used at a visit. Compare patients inside one edition, or use the z-scores.

In [ ]:
print(data["wais_version"].value_counts())

inclusion.groupby("wais_version")[["wais3_ds_total", "wais4_ds_total"]].agg(["count", "mean"]).round(1)

**The tests changed too.** A first battery was used until 2015 to 2018 (fNART, CPT-IP, six elements, doors test, verbal fluency), and other tests came later (SSTICS, six films, errands test). The cell below counts, year by year, the inclusion visits that hold three of them.

In [ ]:
year = inclusion["visit_date"].dt.year

inclusion[["fnart_total", "cptip_dprime_mean", "sstics_total"]].notna().groupby(year).sum().plot(marker="o", figsize=(10, 3.5), title="Inclusion visits that hold the test")
plt.show()

## 8. Treatments

`result.treatments` has one row per patient and medication, with the name as it was written, the molecule, a group, and the start and end dates.

In [ ]:
treatments = result.treatments
print(len(treatments), "medications of", treatments["usubjid"].nunique(), "patients")

figure, (left, right) = plt.subplots(1, 2, figsize=(12, 4))
treatments["treatment_group"].value_counts().sort_values().plot.barh(ax=left, title="Medications per group")
treatments["molecule"].value_counts().head(15).sort_values().plot.barh(ax=right, title="The 15 most frequent molecules")
plt.tight_layout()
plt.show()

What you usually want is simpler: was the patient on clozapine on the day of the visit? These answers are already in `data`, as columns that start with `on_`: 1 for yes, 0 for no, empty when the patient has no list of medications. They are computed from the start and end dates of each medication.

In [ ]:
groups = ["on_antipsychotic_2g", "on_antipsychotic_1g", "on_clozapine", "on_antidepressant", "on_benzodiazepine", "on_anticonvulsant", "on_lithium"]

(inclusion[groups].mean() * 100).round(0).sort_values().plot.barh(title="Patients on each group at inclusion (%)")
plt.show()

## 9. History, comorbidities and family

Yes/no answers are stored as 1 and 0, so the mean of a column is the share of "yes". Always show the number of answers next to it.

These four columns end in `_v0` or `_lifetime`: they are the answers of the inclusion visit, about the whole life (see the traps in section 12).

In [ ]:
answers = ["anxiety_disorder_v0", "alcohol_disorder_v0", "cannabis_disorder_v0", "isf05_lifetime"]

table = inclusion[answers].agg(["count", "mean"]).T
table["percent"] = (table["mean"] * 100).round(1)
table[["count", "percent"]]

The course of the illness is asked at inclusion too: the age at the first psychotic episode and the number of months before the first treatment.

In [ ]:
print(inclusion[["first_episode_age", "untreated_psychosis_months", "n_psychotic_episodes"]].describe().round(1))

inclusion["first_episode_age"].plot.hist(bins=30, title="Age at the first psychotic episode (years)")
plt.show()

The family history comes in two forms. The table `relatives` has one row per relative. The columns of `patients` that start with `fdr_` sum it up for the parents, brothers, sisters and children: 1 when one of them has the disorder, 0 when all of them are answered and none has it, and empty when it cannot be told.

In [ ]:
print(result.relatives["kinship"].value_counts())
print()
print(patients["fdr_schizophrenia"].value_counts(dropna=False))

## 10. Follow patients over time

### A few patients

We look for patients who have a PANSS at five visits or more, and take six of them. `random_state=1` makes it the same six every time; change the number to see others.

The patients are called "Patient 1" to "Patient 6" in the figure, so that no patient number is printed.

In [ ]:
followed = data[data["panss_total"].notna() & data["visit_month"].notna()]

visits_per_patient = followed.groupby("usubjid").size()
often_seen = visits_per_patient[visits_per_patient >= 5].index.to_series()
chosen = often_seen.sample(6, random_state=1).tolist()

print(len(often_seen), "patients have a PANSS at five visits or more")

In [ ]:
figure, (left, right) = plt.subplots(1, 2, figsize=(12, 4), sharex=True)

for number, patient in enumerate(chosen, start=1):
    visits = followed[followed["usubjid"] == patient].sort_values("visit_month")
    left.plot(visits["visit_month"], visits["panss_positive"], marker="o", label=f"Patient {number}")
    right.plot(visits["visit_month"], visits["panss_negative"], marker="o")

left.set_title("PANSS positive scale")
right.set_title("PANSS negative scale")
left.set_xlabel("Months since inclusion")
right.set_xlabel("Months since inclusion")
left.legend()
plt.show()

The whole follow-up of one patient as a table: choose the columns you want to read side by side.

In [ ]:
one = followed[followed["usubjid"] == chosen[0]].sort_values("visit_month")

one[["visit", "visit_date", "age", "panss_total", "cgi_severity", "psp_total", "bmi", "on_clozapine"]]

### The whole cohort

The mean score at inclusion and at each yearly visit. Look at `count` first: fewer and fewer patients are seen.

In [ ]:
yearly = data[data["visit"].isin(["V0", "V1_an", "V2_ans", "V3_ans", "V4_ans", "V5_ans"])]

summary = yearly.groupby("visit_month")[["panss_total", "psp_total"]].agg(["count", "mean"]).round(1)
print(summary)

yearly.groupby("visit_month")[["panss_total", "psp_total"]].mean().plot(marker="o", title="Mean score at each yearly visit")
plt.xlabel("Months since inclusion")
plt.show()

**Be careful with this figure.** The patients seen at three years are not all the patients included: some left, some were included too recently. If those who stay were less ill from the start, the mean goes down even if nobody improves. The cell below compares, at inclusion, the patients who were seen at three years with the others.

In [ ]:
seen_at_3_years = data[data["visit"] == "V3_ans"]["usubjid"]
inclusion["seen_at_3_years"] = inclusion["usubjid"].isin(seen_at_3_years)

inclusion.groupby("seen_at_3_years")[["age", "panss_total", "psp_total"]].agg(["count", "mean"]).round(1)

A fairer way to look at change is to compare each patient with themself. `pivot` puts the two visits side by side, one row per patient, and `dropna()` keeps the patients who have both.

In [ ]:
two_visits = data[data["visit"].isin(["V0", "V1_an"])]
side_by_side = two_visits.pivot(index="usubjid", columns="visit", values="panss_total").dropna()

change = side_by_side["V1_an"] - side_by_side["V0"]
print(len(change), "patients have a PANSS at inclusion and at one year")
print("Mean change:", round(change.mean(), 1), "(below 0 is an improvement)")

change.plot.hist(bins=40, title="Change of the PANSS total between inclusion and one year")
plt.show()

Cognition is measured at inclusion, at three years and at five years, so the same `pivot` works with `V0` and `V3_ans`. The number of words recalled in the CVLT is used here, not its z-score (trap 9 below says why).

In [ ]:
memory = data[data["visit"].isin(["V0", "V3_ans"])].pivot(index="usubjid", columns="visit", values="cvlt_total_1_5").dropna()

print(len(memory), "patients have a CVLT at inclusion and at three years")
(memory["V3_ans"] - memory["V0"]).plot.hist(bins=30, title="Change of the words recalled (CVLT) between inclusion and three years")
plt.show()

## 11. How much can a value be trusted?

While loading, the package repairs what it can and writes down everything it does in `result.flags`. There are three levels:

- `info`: the value was repaired or annotated and can be trusted (a decimal comma, a unit converted, a "less than 5").
- `discrepancy`: the value is kept, but something disagrees with it. Decide for yourself.
- `danger`: the value could not be used. It is empty in `data`, and the text that was typed is kept in the flags.

In [ ]:
flags = result.flags
print(flags["severity"].value_counts())
print()
print(flags[flags["variable"] == "prolactin"]["flag"].value_counts())

For prolactin, `unit_converted` and `unit_inferred` mean that results written in mUI/L were brought to ng/mL; `unit_unknown` means that no unit could be given, so the result is empty. Each flag keeps the text as it was typed (`raw`) and the value now delivered (`value`).

The catalogue also gives, for each variable, how many cells carry each kind of flag.

In [ ]:
result.qc[result.qc["variable"].isin(["prolactin", "crp", "panss_total", "cvlt_total_1_5_z"])][["variable", "n_filled", "n_valid", "n_info", "n_discrepancy", "n_danger", "min", "median", "max"]]

**An example of a doubt.** The z-score of the CVLT learning trials is wrong in the database from 2022 to the middle of 2025: it rises while the number of words recalled does not move. The package removes the impossible values and keeps the others with the flag `period_doubt`. The cell below shows the median of both, year by year.

In [ ]:
visit_year = data["visit_date"].dt.year

data.groupby(visit_year)[["cvlt_total_1_5_z", "cvlt_total_1_5"]].median().round(1).tail(9)

**If you want to be strict**, you can empty the values that carry a `discrepancy` flag before an analysis. The cell below does it for this z-score, in a copy of the table.

In [ ]:
doubted = flags[(flags["variable"] == "cvlt_total_1_5_z") & (flags["severity"] == "discrepancy")][["usubjid", "visit"]].drop_duplicates()
doubted["doubted"] = True

strict = data.merge(doubted, on=["usubjid", "visit"], how="left")
strict.loc[strict["doubted"] == True, "cvlt_total_1_5_z"] = float("nan")

print("CVLT z-scores:", data["cvlt_total_1_5_z"].notna().sum(), "before,", strict["cvlt_total_1_5_z"].notna().sum(), "after")

**The quality-control report.** The package can also write a small website with one page per instrument: the notes, the quality figures and four pictures per variable (by year, by centre). It takes several minutes. Remove the `#` in the cell below to make it, then open `reports/sz/index.html` in a web browser.

In [ ]:
# sz.report(result, "reports/sz")

## 12. The traps of this cohort

Read this list once; it will save you from the most common mistakes.

1. **`_v0` and `_fu` are two different questions.** For comorbidities, medical history, hospitalisations and suicidal behaviour, the inclusion visit asks about the whole life (`_v0`, `_lifetime`) and the later visits ask about the time since the last visit (`_fu`, `_since_visit`, `_12m`). Never put the two in one column.
2. **Editions have their own columns.** WAIS-III and WAIS-IV, for every subtest and for the IQs. `wais_version` says which one a visit used.
3. **Empty does not mean "no".** An empty cell means not asked, not answered or not usable. A "no" is written 0.
4. **Sex, diagnosis and family history are in `patients`.** Join them to the visits with `merge`, as in section 3.
5. **Screening records come before inclusion** and hold an EQ-5D only. Start from `V0`.
6. **Patients leave.** Fewer than half are seen at one year. Every figure over time mixes change and departures (section 10).
7. **Centres differ.** They do not include the same patients and do not fill the forms the same way. Look at your variable per centre before you conclude.
8. **The forms changed over the years**: the tests around 2015 to 2018, part of the medical history in 2022. Always look at your variable per year. The cell below shows one example.
9. **Some scores were wrong in the database.** Three PANSS factor scores, two WAIS-IV indices and the laterality quotient are delivered corrected. Others could only be flagged: the CVLT z-score of 2022 to mid-2025 (section 11) and the Barona expected IQ, whose occupation is most often a copy of the diploma.
10. **Some answers are lost by the export.** No patient is "married or living as a couple", no source of income is a salary, the Calgary scale has no item 8, and `substance_disorder_v0` is empty where the answer was "no": use the column of each substance (`alcohol_disorder_v0`, `cannabis_disorder_v0`...).
11. **Extreme values of measures are brought back** to the 1st and 99th percentile (flag `winsorised`). The typed value is kept in the flags.
12. **Everything is a draft until it is reviewed.** The notes of each instrument and the file `docs/sz/PROGRESS.md` list what is known and what is still an open question.

An example of trap 8. Digestive diseases at inclusion seem to double from 2022. They do not: from 2022 the section is mostly filled for the patients who have such a disease, so there are far fewer answers (`answered`) and almost all the missing ones would be "no".

In [ ]:
digestive = ["inflammatory_bowel_disease_v0", "cirrhosis_v0", "peptic_ulcer_v0", "drug_hepatitis_v0", "diarrhoea_v0", "constipation_v0"]
year = inclusion["visit_date"].dt.year

by_year = pd.DataFrame({
    "inclusions": inclusion.groupby(year).size(),
    "answered": inclusion[digestive].notna().any(axis=1).groupby(year).sum(),
    "with a disease": (inclusion[digestive] == 1).any(axis=1).groupby(year).sum(),
})
by_year["percent of the answered"] = (100 * by_year["with a disease"] / by_year["answered"]).round(0)
by_year["percent of the inclusions"] = (100 * by_year["with a disease"] / by_year["inclusions"]).round(0)
by_year.tail(10)

And trap 7: the same score in each centre. A centre with few patients gives an unstable mean, so look at `count` too.

In [ ]:
inclusion.groupby("siteid")["psp_total"].agg(["count", "mean"]).round(1)

## 13. Build your own table and save it

A typical analysis needs one row per patient with a handful of columns. Choose the visit, choose the columns, join the patient columns, and save.

The file goes to `data/exports/`, which stays on this computer.

In [ ]:
from pathlib import Path

columns = ["usubjid", "visit_date", "siteid", "age", "panss_total", "panss_positive", "panss_negative", "psp_total", "bmi", "on_clozapine", "cannabis_disorder_v0"]

my_table = data[data["visit"] == "V0"][columns]
my_table = my_table.merge(patients[["usubjid", "sex", "diagnosis", "fdr_schizophrenia"]], on="usubjid")

Path("data/exports").mkdir(exist_ok=True)
my_table.to_csv("data/exports/my_first_sz_table.csv", index=False)

print(len(my_table), "patients and", my_table.shape[1], "columns saved")

A first question with this table: do patients with a cannabis use disorder have more positive symptoms at inclusion?

This is a description, not a proof. For a test or a model that takes age, sex and centre into account, open the saved file in your statistics software (R, SPSS, Jamovi) or ask a statistician.

In [ ]:
print(my_table.groupby("cannabis_disorder_v0")["panss_positive"].agg(["count", "mean", "std"]).round(1))

my_table.boxplot(column="panss_positive", by="cannabis_disorder_v0")
plt.show()

## 14. Load only what you need

Loading everything takes half a minute. When you know which instruments you need, name them: it takes a few seconds. The names are in the column `instrument` of the catalogue.

In [ ]:
small = sz.load(EXTRACTION, instruments=["panss", "cgi_egf", "psp"])

print(small.data.shape[1], "columns instead of", data.shape[1])

## 15. Put the bipolar and the schizophrenia cohorts together

The two cohorts are loaded by the same package, and the instruments that both have keep the same names and the same columns. So their tables can be stacked one under the other.

**Step 1: which instruments do both cohorts have?** The cell below compares the lists of instruments. It reads no patient data.

In [ ]:
from facedata import bp

BP_EXTRACTION = "data/bp/BPALL_20261113_2.tsv"     # the bipolar file you received

shared = sorted(set(load_schemas(bp.SCHEMAS)) & set(load_schemas(sz.SCHEMAS)))

print(len(shared), "instruments have the same name in the two cohorts:")
print(", ".join(shared))

**Step 2: load the same instruments from both files.** We take the identity, the CGI and the GAF, the YMRS, the childhood trauma questionnaire, the EQ-5D, weight and height, tobacco, the lifetime suicidal behaviour and the medications.

In [ ]:
wanted = ["identite", "cgi_egf", "ymrs", "ctq", "eq5d", "parametres_physiques", "tobacco", "isf_lifetime", "medications"]

bipolar = bp.load(BP_EXTRACTION, instruments=wanted)
schizophrenia = sz.load(EXTRACTION, instruments=wanted)

print("bipolar:", bipolar.data.shape, " schizophrenia:", schizophrenia.data.shape)

**Step 3: stack the tables** with `pd.concat`. The first column, `cohort`, says where each row comes from.

In [ ]:
both = pd.concat([bipolar.data, schizophrenia.data], ignore_index=True)
both_patients = pd.concat([bipolar.patients, schizophrenia.patients], ignore_index=True)

print(both["cohort"].value_counts())
print()
print(both_patients.groupby("cohort")["diagnosis"].value_counts())

**The key of a patient is `cohort` with `usubjid`.** A patient number is given inside a cohort: the same number can be another person in the other cohort. So every `merge` and every `groupby` of the stacked tables must name both columns.

`patients` also holds `fondacode`, a patient code that is unique across the cohorts and does not change from one extraction to the next. The cell below counts the numbers used in both cohorts, and checks that no `fondacode` appears twice.

In [ ]:
numbers_bp = set(bipolar.patients["usubjid"])
numbers_sz = set(schizophrenia.patients["usubjid"])

print(len(numbers_bp & numbers_sz), "patient numbers are used in both cohorts, for different patients")
print(both_patients["fondacode"].duplicated().sum(), "fondacode appears twice")

**Step 4: join and compare.** The inclusion visit of every patient, with the sex and the diagnosis. Note the two columns in `on=`.

In [ ]:
first = both[both["visit"] == "V0"].merge(both_patients[["cohort", "usubjid", "sex", "diagnosis"]], on=["cohort", "usubjid"])

first.groupby("cohort")[["age", "bmi", "gaf", "cgi_severity", "ymrs_total", "ctq_total", "eq5d_index"]].agg(["count", "mean"]).round(1).T

In [ ]:
first.boxplot(column="gaf", by="diagnosis", rot=30, figsize=(10, 4))
plt.title("Global functioning (GAF) at inclusion, by diagnosis")
plt.suptitle("")
plt.show()

Childhood trauma in the two cohorts: two histograms on the same axes. `density=True` makes them comparable although the cohorts do not have the same size.

In [ ]:
for name, group in first.groupby("cohort"):
    group["ctq_total"].plot.hist(bins=30, alpha=0.5, density=True, label=name)

plt.title("Childhood trauma questionnaire, total score at inclusion")
plt.legend()
plt.show()

Tobacco and treatments, as shares of each cohort. `pd.crosstab` counts the patients for each pair of values; `normalize="index"` turns each row into shares.

In [ ]:
print((pd.crosstab(first["cohort"], first["tobacco_status"], normalize="index") * 100).round(0))

on_groups = ["on_lithium", "on_anticonvulsant", "on_antipsychotic_2g", "on_antipsychotic_1g", "on_antidepressant", "on_benzodiazepine"]
(first.groupby("cohort")[on_groups].mean().T * 100).round(0).plot.barh(title="Patients on each group at inclusion (%)")
plt.show()

**Over time, in both cohorts.** The same move as in section 10, with `cohort` added to the `groupby`. The bipolar cohort also has six-month visits; here we keep the yearly ones.

In [ ]:
yearly_both = both[both["visit"].isin(["V0", "V1_an", "V2_ans", "V3_ans", "V4_ans", "V5_ans"])]

print(yearly_both.groupby(["visit_month", "cohort"])["gaf"].count().unstack())

yearly_both.groupby(["visit_month", "cohort"])["gaf"].mean().unstack().plot(marker="o", title="Mean GAF at each yearly visit")
plt.xlabel("Months since inclusion")
plt.show()

**The medications stack too.** One row per patient and medication, for both cohorts.

In [ ]:
both_treatments = pd.concat([bipolar.treatments, schizophrenia.treatments], ignore_index=True)

top = both_treatments["molecule"].value_counts().head(12).index
pd.crosstab(both_treatments["molecule"], both_treatments["cohort"]).loc[top]

**Three things to check before you compare the cohorts.**

1. **A column that only one cohort has is empty for the other.** `on_clozapine` exists in SZ only; the standing blood pressure in BP only. Count the values per cohort before you use a column.
2. **The same name does not always mean exactly the same form.** The file `facedata/sz/differences.yaml` lists, for each shared instrument, what differs in SZ. The cell below prints it for the instruments we loaded.
3. **The cohorts were not recruited the same way**, and their centres and years differ. A difference between cohorts is a description, not an effect of the diagnosis.

In [ ]:
import yaml

print(both.groupby("cohort")[["gaf", "ctq_total", "on_lithium", "on_clozapine"]].count())
print()

differences = yaml.load(open("facedata/sz/differences.yaml", encoding="utf-8"), Loader=yaml.BaseLoader)
for name in wanted:
    print(name, ":", differences.get(name, "the same in the two cohorts"))

**Save the stacked table**, as in section 13. To stack everything, load both cohorts without `instruments=` and use the same `pd.concat`: it takes about two minutes and gives a table of several thousand columns, most of them empty for one cohort.

In [ ]:
first.to_csv("data/exports/bp_and_sz_inclusion.csv", index=False)

print(len(first), "patients of", first["cohort"].nunique(), "cohorts and", first.shape[1], "columns saved")

## Where to go next

- The notes of each instrument (section 4) and the report (section 11).
- `README.md` for the package, and `docs/sz/PROGRESS.md` for every decision taken about the data and every open question.
- `bipolar-tutorial.ipynb`, the same tour for the bipolar cohort.
- The five moves used in this notebook cover most needs: keep rows with `table[table["column"] == value]`, count with `value_counts()`, summarise with `groupby(...).mean()`, join with `merge`, and draw with `.plot`.

**Before you close: clear the results** (menu *Edit*, *Clear Outputs of All Cells*) if this notebook will be shared.